# Clustered Curve Beta CDF Model

This notebook consumes the CSV produced by `custpaydetails_clustered_cumulative_curves.sql`, fits Beta CDF expected cumulative burn curves, evaluates held-out items, and compares the Beta CDF approach to a pure linear spend model.

## Input and Parameterization

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input.csv |
| Cluster curve rows | 7,631 |
| Items | 875 |
| Train items | 701 |
| Test items | 174 |
| Global Beta alpha | 0.486907 |
| Global Beta beta | 0.742934 |
| Global train MSE | 0.04314304 |
| Proxy label CSV | clustered_curve_proxy_labels.csv |
| Proxy-labeled held-out rows | 1,417 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
if not path.exists():
    path = Path('custpaydetails_clustered_cumulative_curves.csv')
if not path.exists():
    path = Path('ci_item_clustered_cumulative_curves.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Linear Spend Baseline

The pure linear model assumes cumulative spend should equal elapsed time:

```text
expected_cumulative_pct_linear = elapsed_pct
position_delta_linear = actual_cumulative_pct - elapsed_pct
```

This is a useful baseline because it is transparent and easy to explain. It is also too rigid: many items are naturally front-loaded or back-loaded, so a linear curve can systematically flag normal burn shapes as risk.

## Held-Out Model Performance

Errors are cumulative-spend-percent errors. MAE `0.15` means an average absolute error of about 15 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Duration-bucket Beta CDF | 0.1396 | 0.1922 | 0.1094 | 0.3150 | -0.0123 |
| Cluster-count-bucket Beta CDF | 0.1471 | 0.2006 | 0.1094 | 0.3513 | -0.0156 |
| Pooled empirical median curve | 0.1476 | 0.2081 | 0.1048 | 0.3550 | -0.0151 |
| Global Beta CDF | 0.1518 | 0.2068 | 0.1172 | 0.3490 | -0.0131 |
| Linear cumulative spend | 0.1790 | 0.2503 | 0.1300 | 0.4412 | -0.1158 |

## Beta CDF vs Linear: Improvement Summary

| Comparison | Value |
| --- | --- |
| MAE improvement vs linear | 22.01% |
| RMSE improvement vs linear | 23.23% |
| P90 absolute-error improvement vs linear | 28.61% |
| Linear bias | -0.1158 |
| Duration-bucket Beta bias | -0.0123 |

## Error Comparison Plot



## Bucketed Beta CDF Parameters

### Duration Buckets

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 0.515337 | 0.802754 |
| 366-730d | 0.540356 | 0.933287 |
| <=180d | 0.741635 | 0.668742 |
| >730d | 0.567823 | 1.110904 |

### Cluster-Count Buckets

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 0.701546 | 1.166126 |
| 4-6 clusters | 0.212137 | 0.284463 |
| 7-12 clusters | 0.453756 | 0.708690 |
| <=3 clusters | 0.149250 | 0.158773 |

## Curve Performance Plot



## Risk Signal Framing

For active items, the same fitted curve can produce budget-overrun and time-overrun risk signals. These are not final labels without a real authorized budget and schedule; they are position signals.

```text
expected_pct = model(elapsed_pct)
position_delta = actual_cumulative_pct - expected_pct

if position_delta > threshold:
    spending too quickly / budget-overrun risk

if position_delta < -threshold:
    spending too slowly / time-overrun risk
```

For budget overrun projection once a budget is available:

```text
projected_final_spend = actual_cumulative_spend / expected_pct
projected_overrun = projected_final_spend - authorized_budget
```

## Beta vs Linear Risk Signals on Held-Out Data

The table shows how many clustered observations would be flagged by each approach at several position-delta thresholds. `LinearOnly` rows are especially important: these are cases the linear model flags but the duration-bucket Beta curve treats as normal for the observed burn shape.

| Threshold | Signal | Beta count | Linear count | Both | Linear only | Beta only |
| --- | --- | --- | --- | --- | --- | --- |
| 0.10 | spending too quickly / budget-overrun risk | 363 | 660 | 348 | 312 | 15 |
| 0.10 | spending too slowly / time-overrun risk | 379 | 155 | 145 | 10 | 234 |
| 0.15 | spending too quickly / budget-overrun risk | 280 | 542 | 263 | 279 | 17 |
| 0.15 | spending too slowly / time-overrun risk | 283 | 113 | 105 | 8 | 178 |
| 0.25 | spending too quickly / budget-overrun risk | 161 | 331 | 153 | 178 | 8 |
| 0.25 | spending too slowly / time-overrun risk | 99 | 59 | 51 | 8 | 48 |

## Proxy ROC/AUC Setup

True ROC/AUC requires true binary outcome labels. This notebook now consumes retrospective proxy labels from `clustered_curve_proxy_labels.csv`, which is generated by the separate polynomial proxy-label notebook. The Beta CDF and linear models do not create those labels; they only score against them.

These ROC curves compare how well Beta and linear position scores recover the external proxy labels. They are useful for model behavior comparison, but they are not a substitute for true budget/schedule outcome labels.

## Proxy ROC/AUC Summary

| Signal | Model | AUC | Positive labels | Negative labels |
| --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.9823 | 316 | 1,101 |
| fast_spend_proxy | Linear cumulative spend | 0.9789 | 316 | 1,101 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.9770 | 255 | 1,162 |
| slow_spend_proxy | Linear cumulative spend | 0.9452 | 255 | 1,162 |

## Threshold Sweep for Proxy Labels

| Signal | Model | Threshold | Flagged | TP | FP | Precision | Recall/TPR | FPR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.05 | 489 | 316 | 173 | 0.646 | 1.000 | 0.157 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.10 | 363 | 294 | 69 | 0.810 | 0.930 | 0.063 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.15 | 280 | 247 | 33 | 0.882 | 0.782 | 0.030 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.20 | 216 | 203 | 13 | 0.940 | 0.642 | 0.012 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.25 | 161 | 154 | 7 | 0.957 | 0.487 | 0.006 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.30 | 108 | 108 | 0 | 1.000 | 0.342 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.40 | 51 | 51 | 0 | 1.000 | 0.161 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.05 | 772 | 316 | 456 | 0.409 | 1.000 | 0.414 |
| fast_spend_proxy | Linear cumulative spend | 0.10 | 660 | 316 | 344 | 0.479 | 1.000 | 0.312 |
| fast_spend_proxy | Linear cumulative spend | 0.15 | 542 | 316 | 226 | 0.583 | 1.000 | 0.205 |
| fast_spend_proxy | Linear cumulative spend | 0.20 | 431 | 298 | 133 | 0.691 | 0.943 | 0.121 |
| fast_spend_proxy | Linear cumulative spend | 0.25 | 331 | 270 | 61 | 0.816 | 0.854 | 0.055 |
| fast_spend_proxy | Linear cumulative spend | 0.30 | 257 | 238 | 19 | 0.926 | 0.753 | 0.017 |
| fast_spend_proxy | Linear cumulative spend | 0.40 | 162 | 162 | 0 | 1.000 | 0.513 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.05 | 479 | 246 | 233 | 0.514 | 0.965 | 0.201 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.10 | 379 | 236 | 143 | 0.623 | 0.925 | 0.123 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.15 | 283 | 222 | 61 | 0.784 | 0.871 | 0.052 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.20 | 171 | 170 | 1 | 0.994 | 0.667 | 0.001 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.25 | 99 | 99 | 0 | 1.000 | 0.388 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.30 | 55 | 55 | 0 | 1.000 | 0.216 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.40 | 18 | 18 | 0 | 1.000 | 0.071 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.05 | 217 | 177 | 40 | 0.816 | 0.694 | 0.034 |
| slow_spend_proxy | Linear cumulative spend | 0.10 | 155 | 145 | 10 | 0.935 | 0.569 | 0.009 |
| slow_spend_proxy | Linear cumulative spend | 0.15 | 113 | 113 | 0 | 1.000 | 0.443 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.20 | 85 | 85 | 0 | 1.000 | 0.333 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.25 | 59 | 59 | 0 | 1.000 | 0.231 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.30 | 42 | 42 | 0 | 1.000 | 0.165 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.40 | 18 | 18 | 0 | 1.000 | 0.071 | 0.000 |

## Proxy ROC Curves



## Residual Distribution Plot

Residual is `actual cumulative pct - expected cumulative pct`. Positive residuals indicate spending ahead of expected position; negative residuals indicate spending behind expected position.



## Recommendations

Use the duration-bucket Beta CDF as the first expected-position model and keep the linear model as a transparent benchmark. For production alerting:

- Use Beta CDF `position_delta` for primary spend-too-fast and spend-too-slow signals.
- Show the linear delta as a secondary reference because users understand it.
- Alert only when position delta exceeds a threshold and remains elevated across updates.
- For budget-overrun detection, combine position delta with authorized budget and projected final spend.
- For time-overrun detection, combine slow-spend position delta with schedule metadata; slow spending can mean delay, scope removal, or inactive work.